In [1]:
import pandas as pd
url = "https://raw.githubusercontent.com/ogut77/DataScience/master/insurance.csv"
df = pd.read_csv(url)


In [2]:
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


Context in Insurance Data
This dataset is often used to predict charges based on the other variables (age, sex, bmi, children, smoker, region). For example:

Input Variables (X): age, sex, bmi, children, smoker, region (features used to make predictions).

Output Variable (y): charges (what you’re trying to predict).

Describtion of variables
1. Age
Description: The age of the individual (the insured person).
Type: Numerical (integer).
Example Values: 19, 45, 62, etc.
Role in Insurance: Age is a key factor in determining insurance charges. Older individuals often have higher medical costs (and thus higher charges) due to increased health risks.
2. Sex
Description: The gender of the individual.
Type: Categorical (text or binary).
Example Values: "male," "female"
Role in Insurance: Gender can influence insurance charges because health risks and medical expenses may differ between males and females (e.g., pregnancy-related costs for females).
3. BMI (Body Mass Index)
Description: A measure of body fat based on height and weight (calculated as weight in kg divided by height in meters squared).
Type: Numerical (float).
Example Values: 25.3, 30.1, 18.5, etc.
Role in Insurance: Higher BMI often correlates with increased health risks (e.g., obesity-related conditions like diabetes or heart disease), leading to higher insurance charges.
4. Children
Description: The number of children (dependents) covered under the individual’s insurance plan.
Type: Numerical (integer).
Example Values: 0, 1, 3, etc.
Role in Insurance: More children can increase insurance costs slightly, as it may reflect additional healthcare needs, though the effect is often less pronounced than other factors like smoking or age.
5. Smoker
Description: Indicates whether the individual smokes tobacco.
Type: Categorical (text or binary).
Example Values: "yes," "no" .
Role in Insurance: Smoking is a major factor in insurance charges. Smokers typically have much higher medical costs due to risks like lung disease or cancer, so their charges are significantly elevated.
6. Region
Description: The geographic region where the individual lives.
Type: Categorical (text).
Example Values: "northeast," "southeast," "southwest," "northwest" (common in U.S.-based datasets).
Role in Insurance: Charges can vary by region due to differences in healthcare costs, lifestyle factors, or local insurance regulations.
7. Charges
Description: The insurance charges (or premiums/costs) billed to the individual, typically in a currency like USD.
Type: Numerical (float).
Example Values: 1684.52, 11234.89, 32050.23, etc.
Role in Insurance: This is usually the target variable (output) in predictive modeling. It represents the amount the insurance company charges, influenced by all the other columns (age, sex, BMI, etc.).



In [3]:
#1. Check if there is null value in dataset df (5 pt)
df.isnull().sum()

,0
age,0
sex,0
bmi,0
children,0
smoker,0
region,0
charges,0


In [4]:
#2. Assign charges to y  and others to X using df. y is output variable and X is input variables (5 pt)

y = df['charges']
X = df.drop(columns=['charges']) # we need to drop it because now we have charges for y

X.head()

,age,sex,bmi,children,smoker,region
0,19,female,27.900,0,yes,southwest
1,18,male,33.770,1,no,southeast
2,28,male,33.000,3,no,southeast
3,33,male,22.705,0,no,northwest
4,32,male,28.880,0,no,northwest


In [5]:
#3. Use  get_dummies() function from the pandas library to convert categorical variables in a DataFrame (X).
# Drop first drops the first category’s dummy variable to avoid multicollinearity (5 pt)

X = pd.get_dummies(X, drop_first=True)
X.head()

,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27.900,0,False,True,False,False,True
1,18,33.770,1,True,False,False,True,False
2,28,33.000,3,True,False,False,True,False
3,33,22.705,0,True,False,True,False,False
4,32,28.880,0,True,False,True,False,False


In [6]:
#Use following methods for the evaluation on test and train data
def evalmetric(y,ypred):
 from scipy.stats import pearsonr
 import numpy as np
 e = y - ypred
 mse_f = np.mean(e**2)
 rmse_f = np.sqrt(mse_f)
 mae_f = np.mean(abs(e))
 mape_f = 100*np.mean(abs(e/y))
 crl, _ = pearsonr(y, ypred)
 r2_f = crl*crl
 print("MSE:", mse_f)
 print("RMSE:", rmse_f)
 print("MAE:",mae_f)
 print("MAPE:",mape_f)
 print("R-Squared:", round(r2_f, 4))


In [7]:
#4.Get the correlation between X variables and y variables.(5 pt)

temp_df = pd.concat([X, y], axis=1)

correlations = temp_df.corr()['charges'].sort_values(ascending=False) # here we ondly select the charges column

print(correlations)

charges             1.000000
smoker_yes          0.787251
age                 0.299008
bmi                 0.198341
region_southeast    0.073982
children            0.067998
sex_male            0.057292
region_northwest   -0.039905
region_southwest   -0.043210
Name: charges, dtype: float64


In [8]:
#5.Split a dataset into 25%  of data as test data  and 75% of data as training data ( pt)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42) # 25% of data to the test and trainings 75%

print(f"Training set size: {X_train.shape[0]} rows")
print(f"Testing set size: {X_test.shape[0]} rows")

Training set size: 1003 rows
Testing set size: 335 rows


In [9]:
#6. Using Decision Tree and Linear Regression methods, compare the performance results on both test and training data
#to determine which one is more likely to overfit and which is more likely to underfit.
# Do you think that Lasso and Ridge regularization are more likely to improve the results of Linear model test data,
# or would Random Forest or Boosting methods are more likely to improve the results of Decison tree test data?
#Explain your reasoning.(35 pt)

from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression() # Linear Regression
lr_model.fit(X_train, y_train)

print("Linear Regression: Train Performance")
evalmetric(y_train, lr_model.predict(X_train))
print("\nLinear Regression: Test Performance")
evalmetric(y_test, lr_model.predict(X_test))

print("\n" + "="*30 + "\n")

dt_model = DecisionTreeRegressor(random_state=42) # Desicion Tree
dt_model.fit(X_train, y_train)

print("Decision Tree: Train Performance")
evalmetric(y_train, dt_model.predict(X_train))
print("\nDecision Tree: Test Performance")
evalmetric(y_test, dt_model.predict(X_test))

# Q1
# Based on the provided metrics, the Decision Tree is more likely to overfit because it shows nearly perfect performance on the training data (R^2 = 0.9987$).
# but drops significantly on the test data (R^2 = 0.7805$), indicating it has memorized the noise.
# In contrast, Linear Regression is more likely to underfit because its training performance is relatively low (R^2 = 0.745$), suggesting the model is too simple
# to capture the underlying complexity of the dataset.

# Q2
# Lasso and Ridge regularization are unlikely to see massive gains here because the Linear Regression model isn't currently overfitting
# (the test results are actually slightly better than the training results). While these methods help by penalizing large coefficients to reduce variance,
# they are most effective when the model is overly complex

# Q3
# Random Forest and Boosting methods are more likely to significantly improve results for the Decision Tree test data.
# These ensemble techniques are specifically designed to reduce the high variance and overfitting seen in single trees by averaging multiple trees or learning
# from previous errors. Given the huge gap between the Decision Tree’s training and test performance,
# these methods would effectively improve model's ability to generalize.

Linear Regression: Train Performance
MSE: 37004502.18409475
RMSE: 6083.132596294014
MAE: 4183.153367011969
MAPE: 42.26867490005659
R-Squared: 0.745

Linear Regression: Test Performance
MSE: 35117755.73613632
RMSE: 5926.023602394469
MAE: 4243.654116653137
MAPE: 44.468185116980976
R-Squared: 0.7676


Decision Tree: Train Performance
MSE: 182648.17106092346
RMSE: 427.37357318969015
MAE: 19.084122482552342
MAPE: 0.47775094523744194
R-Squared: 0.9987

Decision Tree: Test Performance
MSE: 36580230.16400095
RMSE: 6048.159237652473
MAE: 2693.294304847761
MAPE: 29.869257985320623
R-Squared: 0.7805


In [ ]:
#7. Explain performance of linear regressin on test data
# using  Root mean squared error, mean absolute error, mean absolute percentage error and R2 metric (10 pt)

# 1. RMSE (Root Mean Squared Error):
# Mainly it penalizes large errors. If high, the model is making big mistakes on outliers

# 2. MAE (Mean Absolute Error):
# MAE is the average distance between your prediction and the actual bill. So the difference shows by how much your prediction is off

# 3. MAPE (Mean Absolute Percentage Error):
# Shows error as a percentage. Helps understand accuracy relative to the actual bill size.

# 4. R-Squared (R2):
# The percentage of variance explained. An R2 of 0.75 means the model accounts for 75% of the factors affecting insurance charges.

In [11]:
!pip install xgboost lightgbm catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [14]:
#8. Use Random Forest and Boosting methods (XGBoost, LightGBM, and CatBoost)
#to obtain the evaluation scores on  test data.
#Which Boosting technique yielded the best performance on the test data based on the R² metric?
#Did you achieve a better result compared to Random Forest on the test data based on the R² metric?
#If there is improvement on Random forest or boosting methods over decison tree, explain  (30 pt)

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

models = { # all the models
    "Random Forest": RandomForestRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42),
    "LightGBM": LGBMRegressor(random_state=42),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=0)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"\n--- {name} Test Performance ---")
    evalmetric(y_test, y_pred)

# Q1
# In most cases with this specific insurance dataset, CatBoost or XGBoost typically yields the highest R^{2} score
# CatBoost is particularly good here because it handles the categorical structures such as region and smoker status very efficiently.

# Q2
# Usually, Yes. While Random Forest is excellent at reducing variance, Boosting techniques, XGBoost and CatBoost reduce both bias and variance by
# learning from the mistakes of previous iterations. This typically leads to a higher R^{2} score on the test set.

# Q3
# Single Decision Trees often overfit by memorizing the training data, but Random Forest fixes this by averaging multiple trees to cancel out random noise and variance.
# Boosting methods (XGBoost, LightGBM, CatBoost) further improve performance by building trees sequentially, with each new tree specifically correcting
# the errors of the previous ones. This ensemble approach effectively captures complex, non-linear relationships.


--- Random Forest Test Performance ---
MSE: 23114410.14038345
RMSE: 4807.7448081593775
MAE: 2653.61478095801
MAPE: 30.27386314379988
R-Squared: 0.8506

--- XGBoost Test Performance ---
MSE: 26433443.13176504
RMSE: 5141.346431798293
MAE: 2957.213261792119
MAPE: 34.57266690344694
R-Squared: 0.8301
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000182 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 319
[LightGBM] [Info] Number of data points in the train set: 1003, number of used features: 8
[LightGBM] [Info] Start training from score 13267.935814

--- LightGBM Test Performance ---
MSE: 22005117.949021608
RMSE: 4690.961303296117
MAE: 2700.720352413941
MAPE: 33.101463593940274
R-Squared: 0.8553

--- CatBoost Test Performance ---
MSE: 21340838.308785602
RMSE: 4619.614519501124
MAE: 2608.117710576109
MAPE: 30.82075085150296
R-Squared: 0.8589
